In [84]:
import numpy as np
import sklearn 
from matplotlib import pyplot as plt
import pandas as pd

In [85]:
titanic=sklearn.datasets.fetch_openml('titanic', version=1)
print('len:',len(titanic['data']))
titanic.frame.info()

len: 1309
<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   pclass     1309 non-null   int64   
 1   survived   1309 non-null   category
 2   name       1309 non-null   str     
 3   sex        1309 non-null   category
 4   age        1046 non-null   float64 
 5   sibsp      1309 non-null   int64   
 6   parch      1309 non-null   int64   
 7   ticket     1309 non-null   str     
 8   fare       1308 non-null   float64 
 9   cabin      295 non-null    str     
 10  embarked   1307 non-null   category
 11  boat       486 non-null    str     
 12  body       121 non-null    float64 
 13  home.dest  745 non-null    str     
dtypes: category(3), float64(3), int64(3), str(5)
memory usage: 116.6 KB


In [86]:
for i in titanic.frame.columns.tolist():
    print(i,titanic.frame[i].nunique())
# попробовать добавить tiket
data=titanic['data']
data=data.drop(['body','boat','name','ticket','home.dest'],axis=1)
data['age']=data['age'].fillna(0,inplace=True)
data['fare']=data['fare'].fillna(0,inplace=True)
data['cabin']=data['cabin'].fillna('U',inplace=True)
data['embarked']=data['embarked'].fillna('S')
data['cabin']=data['cabin'].str[0]



for i in data:
    print(i,end='|')
data

pclass 3
survived 2
name 1307
sex 2
age 98
sibsp 7
parch 8
ticket 929
fare 281
cabin 186
embarked 3
boat 27
body 121
home.dest 369
pclass|sex|age|sibsp|parch|fare|cabin|embarked|

C:\Users\user\AppData\Local\Temp\ipykernel_12388\2149467774.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  data['age']=data['age'].fillna(0,inplace=True)
C:\Users\user\AppData\Local\Temp\ipykernel_12388\2149467774.py:7: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment usi

,pclass,sex,age,sibsp,parch,fare,cabin,embarked
0,1,female,29.0000,0,0,211.3375,B,S
1,1,male,0.9167,1,2,151.5500,C,S
2,1,female,2.0000,1,2,151.5500,C,S
3,1,male,30.0000,1,2,151.5500,C,S
4,1,female,25.0000,1,2,151.5500,C,S
...,...,...,...,...,...,...,...,...
1304,3,female,14.5000,1,0,14.4542,U,C
1305,3,female,0.0000,1,0,14.4542,U,C
1306,3,male,26.5000,0,0,7.2250,U,C
1307,3,male,27.0000,0,0,7.2250,U,C


In [87]:
data_train=data.sample(frac=0.7,random_state=1)
data_val=data.drop(data_train.index).sample(frac=1.0,random_state=2)


y_train=titanic['target'].sample(frac=0.7,random_state=1)
y_val=titanic['target'].drop(y_train.index).sample(frac=1.0,random_state=2)



max_age=data_train['age'].max()
max_sibsp=data_train['sibsp'].max()
max_parch=data_train['parch'].max()
max_fare=data_train['fare'].max()

data_train['age'] /=max_age
data_train['sibsp']/=max_sibsp
data_train['parch']/=max_parch
data_train['fare']/=max_fare

data_val['age']/=max_age
data_val['sibsp']/=max_sibsp
data_val['parch']/= max_parch
data_val['fare']/=max_fare

data_train=pd.get_dummies(data_train,columns=['pclass','sex','cabin','embarked'],dtype=int)
data_val=pd.get_dummies(data_val,columns=['pclass','sex','cabin','embarked'],dtype=int)


In [90]:
data_train

,age,sibsp,parch,fare,pclass_1,pclass_2,pclass_3,sex_female,sex_male,cabin_A,...,cabin_C,cabin_D,cabin_E,cabin_F,cabin_G,cabin_T,cabin_U,embarked_C,embarked_Q,embarked_S
201,0.6750,0.000,0.000000,0.101229,1,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,1
115,0.8000,0.125,0.444444,0.513342,1,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,1
255,0.3000,0.000,0.000000,0.135265,1,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,0
1103,0.0250,0.500,0.111111,0.077465,0,0,1,0,1,0,...,0,0,0,0,0,0,1,0,0,1
195,0.2000,0.000,0.000000,0.168837,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
787,0.2000,0.000,0.000000,0.015176,0,0,1,0,1,0,...,0,0,0,0,0,0,1,0,0,1
501,0.1625,0.000,0.111111,0.038061,0,1,0,1,0,0,...,0,0,0,0,0,0,1,0,0,1
816,0.0000,0.000,0.000000,0.015412,0,0,1,0,1,0,...,0,0,0,0,0,0,1,1,0,0
1231,0.3625,0.125,0.111111,0.020421,0,0,1,1,0,0,...,0,0,0,0,1,0,0,0,0,1
